<a href="https://colab.research.google.com/github/tsenga2/ibes-japan/blob/main/ConsolidateData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U -q PyDrive
from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

In [ ]:
import numpy as np
from pandas import Series,DataFrame
import pandas as pd
from numpy import nan
import os

In [ ]:
f = drive.CreateFile({'id': '1nrnCMmOBVU6dWWmW4hKVmSu-D8AWvxjp'})
DBJ_list = drive.ListFile().GetList()[0]['id']
file_list = drive.ListFile({'q': '"1nrnCMmOBVU6dWWmW4hKVmSu-D8AWvxjp" in parents and trashed = false'.format(DBJ_list)}).GetList()
for f in file_list:
     print(f['title'], f['id'])

In [ ]:
for f in file_list:
  if f['title'].isalpha() == False:
     fileDownloaded = drive.CreateFile({'id': f['id']})
     fileDownloaded.GetContentFile(f['title'])
  else: pass

In [ ]:
os.rename('連結キャッシュフロー計算書.txt','conso_cashflow')
#os.rename('連結セグメント情報.txt','conso_segment') manually rename the file and take. txt off
os.rename('連結会社概況.txt','conso_kaisha')
os.rename('連結株主資本等変動計算書.txt','conso_hendou')
os.rename('連結事業状況.txt','conso_overview')
os.rename('連結損益計算書.txt','conso_PL')
os.rename('連結貸借対照表1.txt','conso_BS1')
os.rename('連結貸借対照表2.txt','conso_BS2')
os.rename('連結注記事項.txt','conso_note')

In [ ]:
dat1 =pd.read_table('conso_cashflow', sep=",", index_col=2)
dat2 =pd.read_table('conso_segment', sep=",", index_col=2)
dat3 =pd.read_table('conso_kaisha', sep=",", index_col=2)
dat4 =pd.read_table('conso_hendou', sep=",", index_col=2)
dat5 =pd.read_table('conso_overview', sep=",", index_col=2)
dat6 =pd.read_table('conso_PL', sep=",", index_col=2)
dat7 =pd.read_table('conso_BS1', sep=",", index_col=2)
dat8 =pd.read_table('conso_BS2', sep=",", index_col=2)
dat9 =pd.read_table('conso_note', sep=",", index_col=2)

dat10 = pd.concat([dat1,dat2,dat3,dat4,dat5,dat6,dat7,dat8,dat9],axis=1)


dat10 = dat10.T.drop_duplicates().T


dat10 = dat10.rename(columns={'会社ｺｰﾄﾞ':'Comp_code'})
dat10 = dat10.rename(columns={'株式銘柄ｺｰﾄﾞ':'Stock_code'})
dat10 = dat10.rename(columns={'決算期':'ap'})


col=dat10.columns.tolist()
p=col[:3]
q=col[3:]
q=[t[:10] for t in q]
col=p+q
dat10.columns=col



In [ ]:
ap = pd.DataFrame(dat10['ap'])

fileDownloaded = drive.CreateFile({'id': '1pjh__ax1Dim35oJFjjVx1fN-ApR9zpfY'})
fileDownloaded.GetContentFile('リサーチ項目（連結）.csv')
dat11 = pd.read_csv('リサーチ項目（連結）.csv').T
dat11.columns = dat11.iloc[1]
dat11 = dat11.iloc[0:1]
common_columns = dat10.columns.intersection(dat11.columns)
dat12 = pd.concat([dat10[common_columns], dat11[common_columns]], ignore_index=False)
dat12.dropna(subset=['略称'], axis=1, inplace=True)
dat12.columns = dat12.loc['略称']
dat12.drop(dat12.index[-1], inplace=True)
jbd = pd.concat([dat12, ap],axis=1)
jbd.index.names = ['Stock_code']


jbd.tail()

In [ ]:
jbd.to_csv('renketsu.csv', index=True, encoding='utf-8')

from google.colab import files

# ファイルをダウンロード
files.download('renketsu.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>